# 🧊 SIH2659: Antarctic Iceberg Trajectory Prediction Pipeline
### End-to-End Leakage-Safe Machine Learning in Google Colab

This notebook trains and evaluates classical machine learning models (Persistence Baselines, Random Forest, XGBoost) to predict the 1-step-ahead displacement (`target_delta_lat`, `target_delta_lon`) of Antarctic icebergs using multi-sensor trajectory histories and environmental forcing fields (GLORYS ocean currents, sea-ice concentration, bathymetry).

**Key Integrity Rules:**
- Strict grouping split by `iceberg_id` (Zero leakage across splits)
- Strictly causal features observed at time $t$ (No future leakage)
- Evaluation in Haversine Great-Circle distance (km)

## 1. Mount Google Drive & Configure Paths

In [ ]:
import os
import sys

# Set your Google Drive or local working directory here:
PROJECT_DIR = "/content/drive/MyDrive/IcebergProject"

# Check if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    print(f"Mounted Google Drive. Working directory: {os.getcwd()}")
except Exception as e:
    print(f"Not in Colab or Drive already mounted ({e}). Current directory: {os.getcwd()}")

## 2. Install & Verify Dependencies

In [ ]:
!pip install --quiet pyarrow scikit-learn xgboost matplotlib pandas numpy

import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import joblib

print("=" * 60)
print("ENVIRONMENT & SOFTWARE VERSIONS")
print("=" * 60)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"Pandas Version       : {pd.__version__}")
print(f"NumPy Version        : {np.__version__}")
print(f"Scikit-Learn Version : {sklearn.__version__}")

try:
    import xgboost as xgb
    print(f"XGBoost Version      : {xgb.__version__}")
except ImportError:
    print("XGBoost Version      : NOT INSTALLED")

## 3. Load & Verify Datasets
Loads `train.parquet`, `validation.parquet`, `test.parquet` (or loads `ml_dataset_final.parquet` and performs iceberg-level splitting if splits are not present).

In [ ]:
train_path = os.path.join(PROJECT_DIR, "train.parquet")
val_path   = os.path.join(PROJECT_DIR, "validation.parquet")
test_path  = os.path.join(PROJECT_DIR, "test.parquet")
full_path  = os.path.join(PROJECT_DIR, "ml_dataset_final.parquet")

if os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(test_path):
    df_train = pd.read_parquet(train_path)
    df_val   = pd.read_parquet(val_path)
    df_test  = pd.read_parquet(test_path)
    print("Loaded pre-split datasets successfully.")
else:
    print("Splits not found, loading full dataset and splitting by iceberg_id...")
    df = pd.read_parquet(full_path)
    unique_icebergs = sorted(df["iceberg_id"].unique())
    rng = np.random.RandomState(42)
    shuffled = rng.permutation(unique_icebergs)
    n_tr = int(round(0.7 * len(unique_icebergs)))
    n_va = int(round(0.15 * len(unique_icebergs)))
    tr_ids = set(shuffled[:n_tr])
    va_ids = set(shuffled[n_tr:n_tr+n_va])
    te_ids = set(shuffled[n_tr+n_va:])
    df_train = df[df["iceberg_id"].isin(tr_ids)].reset_index(drop=True)
    df_val   = df[df["iceberg_id"].isin(va_ids)].reset_index(drop=True)
    df_test  = df[df["iceberg_id"].isin(te_ids)].reset_index(drop=True)

print(f"Train observations : {len(df_train):,} ({df_train['iceberg_id'].nunique()} icebergs)")
print(f"Val observations   : {len(df_val):,} ({df_val['iceberg_id'].nunique()} icebergs)")
print(f"Test observations  : {len(df_test):,} ({df_test['iceberg_id'].nunique()} icebergs)")

# Verify zero leakage across splits
tr_set = set(df_train["iceberg_id"])
te_set = set(df_test["iceberg_id"])
assert len(tr_set & te_set) == 0, "LEAKAGE DETECTED! Overlap between Train and Test icebergs."
print("Verified: Zero iceberg overlap between Train and Test sets.")

## 4. Define Feature Space & Fit Imputation Pipeline

In [ ]:
from sklearn.impute import SimpleImputer

FEATURE_COLS = [
    "lat", "lon", "latitude_rad", "longitude_rad",
    "month", "day_of_year", "sin_day_of_year", "cos_day_of_year", "future_dt_days",
    "previous_delta_lat", "previous_delta_lon", "previous_speed_kmday", "previous_direction",
    "previous_dt_days", "delta_lat_lag2", "delta_lon_lag2", "speed_lag2",
    "glorys_uo_ms", "glorys_vo_ms", "glorys_current_speed_ms", "glorys_current_dir",
    "glorys_so", "glorys_bottomT", "seaice_conc_pct", "ocean_depth_m",
    "wind_u10_ms", "wind_v10_ms", "wind_speed_ms", "wind_dir"
]

# Imputer fit ONLY on training set
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(df_train[FEATURE_COLS])
X_val   = imputer.transform(df_val[FEATURE_COLS])
X_test  = imputer.transform(df_test[FEATURE_COLS])

y_train_lat = df_train["target_delta_lat"].values
y_train_lon = df_train["target_delta_lon"].values
y_val_lat   = df_val["target_delta_lat"].values
y_val_lon   = df_val["target_delta_lon"].values
y_test_lat  = df_test["target_delta_lat"].values
y_test_lon  = df_test["target_delta_lon"].values

print(f"Feature matrix shape: {X_train.shape} with {len(FEATURE_COLS)} features.")

## 5. Metric Utilities (Haversine Geographic Distance)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp/2.0)**2 + np.cos(p1) * np.cos(p2) * np.sin(dl/2.0)**2
    return R * 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))

def evaluate(df, pred_dlat, pred_dlon, name):
    actual_lat = df["target_lat"].values
    actual_lon = df["target_lon"].values
    pred_lat = df["lat"].values + pred_dlat
    pred_lon = df["lon"].values + pred_dlon
    pred_lon = ((pred_lon + 180.0) % 360.0) - 180.0

    err_km = haversine_km(actual_lat, actual_lon, pred_lat, pred_lon)
    mae_lat = np.mean(np.abs(pred_dlat - df["target_delta_lat"].values))
    mae_lon = np.mean(np.abs(pred_dlon - df["target_delta_lon"].values))
    return {
        "Model": name,
        "MAE delta_lat": round(float(mae_lat), 5),
        "MAE delta_lon": round(float(mae_lon), 5),
        "Mean Err (km)": round(float(np.mean(err_km)), 2),
        "Median Err (km)": round(float(np.median(err_km)), 2),
        "P90 Err (km)": round(float(np.percentile(err_km, 90)), 2),
        "Max Err (km)": round(float(np.max(err_km)), 2)
    }

## 6. Train Baselines & Machine Learning Models

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

results = []

# 1. Zero Movement
p_lat_zero = np.zeros(len(df_test))
p_lon_zero = np.zeros(len(df_test))
results.append(evaluate(df_test, p_lat_zero, p_lon_zero, "Zero-Movement Persistence"))

# 2. Kinematic Persistence
p_lat_kin = df_test["previous_delta_lat"].fillna(0.0).values
p_lon_kin = df_test["previous_delta_lon"].fillna(0.0).values
results.append(evaluate(df_test, p_lat_kin, p_lon_kin, "Kinematic Persistence"))

# 3. Linear Regression
lr_lat = LinearRegression().fit(X_train, y_train_lat)
lr_lon = LinearRegression().fit(X_train, y_train_lon)
results.append(evaluate(df_test, lr_lat.predict(X_test), lr_lon.predict(X_test), "Linear Regression"))

# 4. Random Forest
print("Training Random Forest models...")
rf_lat = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_lon = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_lat.fit(X_train, y_train_lat)
rf_lon.fit(X_train, y_train_lon)
pred_rf_lat = rf_lat.predict(X_test)
pred_rf_lon = rf_lon.predict(X_test)
results.append(evaluate(df_test, pred_rf_lat, pred_rf_lon, "Random Forest"))

# 5. XGBoost (if installed)
try:
    import xgboost as xgb
    print("Training XGBoost models with early stopping...")
    xgb_lat = xgb.XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=5, subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=25, eval_metric='mae', random_state=42, n_jobs=-1)
    xgb_lon = xgb.XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=5, subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=25, eval_metric='mae', random_state=42, n_jobs=-1)
    xgb_lat.fit(X_train, y_train_lat, eval_set=[(X_val, y_val_lat)], verbose=False)
    xgb_lon.fit(X_train, y_train_lon, eval_set=[(X_val, y_val_lon)], verbose=False)
    pred_xgb_lat = xgb_lat.predict(X_test)
    pred_xgb_lon = xgb_lon.predict(X_test)
    results.append(evaluate(df_test, pred_xgb_lat, pred_xgb_lon, "XGBoost"))
except Exception as e:
    print(f"XGBoost skipped: {e}")

# Summary Table
df_res = pd.DataFrame(results)
print("\n" + "=" * 70)
print("TEST SET PERFORMANCE COMPARISON")
print("=" * 70)
print(df_res.to_string(index=False))

## 7. Plot Feature Importance

In [ ]:
mean_rf_imp = (rf_lat.feature_importances_ + rf_lon.feature_importances_) / 2.0
df_feat_imp = pd.DataFrame({"feature": FEATURE_COLS, "importance": mean_rf_imp}).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(df_feat_imp.tail(15)["feature"], df_feat_imp.tail(15)["importance"], color="#1f77b4", edgecolor="black")
plt.xlabel("Mean Relative Feature Importance")
plt.title("Top 15 Most Important Features for Next-Step Iceberg Drift")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 8. Visualize Predicted vs Actual Trajectories (Unseen Icebergs)

In [ ]:
top_test_ibs = df_test["iceberg_id"].value_counts().head(3).index
df_test["pred_dlat"] = pred_rf_lat
df_test["pred_dlon"] = pred_rf_lon

for ib in top_test_ibs:
    sub = df_test[df_test["iceberg_id"] == ib].sort_values("date")
    plt.figure(figsize=(8, 6))
    plt.plot(sub["lon"], sub["lat"], "b-o", label="Actual Trajectory", alpha=0.85)
    plt.plot(sub["lon"] + sub["pred_dlon"], sub["lat"] + sub["pred_dlat"], "r--^", label="Random Forest 1-Step Ahead", alpha=0.75)
    plt.title(f"Iceberg Track: {ib} (Actual vs Predicted)")
    plt.xlabel("Longitude (°E/W)")
    plt.ylabel("Latitude (°S)")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 9. Save Trained Models to Google Drive

In [ ]:
out_model_dir = os.path.join(PROJECT_DIR, "trained_models")
os.makedirs(out_model_dir, exist_ok=True)

joblib.dump(rf_lat, os.path.join(out_model_dir, "random_forest_lat.joblib"))
joblib.dump(rf_lon, os.path.join(out_model_dir, "random_forest_lon.joblib"))
joblib.dump(imputer, os.path.join(out_model_dir, "rf_imputer.joblib"))
df_res.to_csv(os.path.join(out_model_dir, "test_evaluation_results.csv"), index=False)

print(f"Models and test results saved to: {out_model_dir}")